# Ratebook Optimiser Demo

Coordinate descent over rating factors. The solver finds per-group
multiplicative adjustments (e.g., region, age_band, vehicle_type)
that optimise the portfolio subject to constraints.

This is the core ratebook use-case: discovering factor tables that can
be loaded into a rating engine.

In [ ]:
import math
import time

import polars as pl
import price_contour as pc
from price_contour.ratebook import RatebookOptimiser, RatebookResult

print(f"price_contour {pc.__version__}")

## 1. Generate scored data + rating factors

We generate:
- A scored DataFrame with 1,000 quotes × 11 multiplier steps
- A factors DataFrame with per-quote rating factor values (region, age_band, vehicle_type)

In [ ]:
N_QUOTES = 1_000
N_STEPS = 11


def make_scored_df(n_quotes=N_QUOTES, n_steps=N_STEPS):
    """Generate synthetic scored DataFrame."""
    mults = [0.80 + 0.04 * j for j in range(n_steps)]
    rows = []
    for q in range(n_quotes):
        elasticity = 1.5 + 3.5 * q / n_quotes
        base = 80.0 + 40.0 * q / n_quotes
        for j, mult in enumerate(mults):
            conversion = 1.0 / (1.0 + math.exp(elasticity * (mult - 1.0)))
            rows.append({
                "quote_id": f"Q{q:05d}",
                "scenario_step": j,
                "multiplier": mult,
                "expected_income": base * mult * conversion,
                "volume": conversion,
                "loss_ratio": 0.6 / mult * (1.0 + 0.1 * (mult - 1.0)),
            })
    return pl.DataFrame(
        rows,
        schema={
            "quote_id": pl.Utf8,
            "scenario_step": pl.Int32,
            "multiplier": pl.Float32,
            "expected_income": pl.Float32,
            "volume": pl.Float32,
            "loss_ratio": pl.Float32,
        },
    )


df = make_scored_df()
print(f"Scored data: {df.shape}  ({df['quote_id'].n_unique()} quotes × {N_STEPS} steps)")
df.head()

In [ ]:
import random

random.seed(42)

regions = ["London", "South East", "Midlands", "North", "Scotland", "Wales"]
age_bands = ["17-25", "26-35", "36-50", "51-65", "66+"]
vehicle_types = ["Hatchback", "Saloon", "SUV", "Van"]

factors = pl.DataFrame({
    "region": [random.choice(regions) for _ in range(N_QUOTES)],
    "age_band": [random.choice(age_bands) for _ in range(N_QUOTES)],
    "vehicle_type": [random.choice(vehicle_types) for _ in range(N_QUOTES)],
})

print(f"Factors: {factors.shape}")
print(f"\nRegion distribution:")
print(factors["region"].value_counts().sort("region"))
print(f"\nAge band distribution:")
print(factors["age_band"].value_counts().sort("age_band"))
print(f"\nVehicle type distribution:")
print(factors["vehicle_type"].value_counts().sort("vehicle_type"))

## 2. Single-factor optimisation

Optimise region factors only. The solver finds a multiplicative adjustment
for each region that maximises expected income while maintaining volume ≥ 90% of baseline.

In [ ]:
opt_single = RatebookOptimiser(
    objective="expected_income",
    constraints={"volume": {"min": 0.90}},
    factor_columns=[["region"]],
    candidate_min=0.80,
    candidate_max=1.20,
    candidate_steps=50,
    max_cd_iterations=1,
    max_iter=100,
)

t0 = time.perf_counter()
result_single = opt_single.solve(df, factors)
elapsed = time.perf_counter() - t0

print(f"Single-factor solve: {elapsed:.2f}s")
print(f"  Total objective:  {result_single.total_objective:,.2f}")
print(f"  Baseline:         {result_single.baseline_objective:,.2f}")
if result_single.baseline_objective != 0:
    uplift = (result_single.total_objective - result_single.baseline_objective) / abs(result_single.baseline_objective) * 100
    print(f"  Uplift:           {uplift:+.2f}%")
print(f"  CD iterations:    {result_single.cd_iterations}")
print(f"  Converged:        {result_single.converged}")
print(f"  Clamp rate:       {result_single.clamp_rate:.4f}")
print(f"  Lambdas:          {result_single.lambdas}")

In [ ]:
print("Region factor table:")
for level, factor in sorted(result_single.factor_tables["region"].items()):
    print(f"  {level:15s}  {factor:.4f}")

## 3. Multi-factor optimisation

Optimise all three factors via coordinate descent. Each CD iteration
cycles through region → age_band → vehicle_type, updating one factor
at a time while holding others fixed.

In [ ]:
opt_multi = RatebookOptimiser(
    objective="expected_income",
    constraints={"volume": {"min": 0.90}},
    factor_columns=[["region"], ["age_band"], ["vehicle_type"]],
    candidate_min=0.80,
    candidate_max=1.20,
    candidate_steps=50,
    max_cd_iterations=5,
    cd_tolerance=1e-3,
    max_iter=100,
)

t0 = time.perf_counter()
result_multi = opt_multi.solve(df, factors)
elapsed = time.perf_counter() - t0

print(f"Multi-factor solve: {elapsed:.2f}s")
print(f"  Total objective:  {result_multi.total_objective:,.2f}")
print(f"  Baseline:         {result_multi.baseline_objective:,.2f}")
if result_multi.baseline_objective != 0:
    uplift = (result_multi.total_objective - result_multi.baseline_objective) / abs(result_multi.baseline_objective) * 100
    print(f"  Uplift:           {uplift:+.2f}%")
print(f"  CD iterations:    {result_multi.cd_iterations}")
print(f"  Converged:        {result_multi.converged}")
print(f"  Clamp rate:       {result_multi.clamp_rate:.4f}")

In [ ]:
for factor_name, table in sorted(result_multi.factor_tables.items()):
    print(f"\n{factor_name} factor table:")
    for level, factor in sorted(table.items()):
        print(f"  {level:15s}  {factor:.4f}")

## 4. Rating entries

Convert factor tables to DataFrames suitable for loading into a rating engine.

In [ ]:
entries = result_multi.to_rating_entries()

for name, entry_df in sorted(entries.items()):
    print(f"\n=== {name} ===")
    print(entry_df)

## 5. Two constraints: volume + loss ratio

In [ ]:
opt_2c = RatebookOptimiser(
    objective="expected_income",
    constraints={
        "volume": {"min": 0.90},
        "loss_ratio": {"max": 1.05},
    },
    factor_columns=[["region"], ["age_band"]],
    candidate_min=0.80,
    candidate_max=1.20,
    candidate_steps=50,
    max_cd_iterations=3,
    max_iter=100,
)

t0 = time.perf_counter()
result_2c = opt_2c.solve(df, factors)
elapsed = time.perf_counter() - t0

print(f"Two-constraint solve: {elapsed:.2f}s")
print(f"  Objective:        {result_2c.total_objective:,.2f}")
print(f"  Constraints:      {result_2c.total_constraints}")
print(f"  Lambdas:          {result_2c.lambdas}")
print(f"  CD iterations:    {result_2c.cd_iterations}")

for name, table in sorted(result_2c.factor_tables.items()):
    print(f"\n{name}:")
    for level, factor in sorted(table.items()):
        print(f"  {level:15s}  {factor:.4f}")

## 6. Auto-discovery

If `factor_columns` is not specified, the solver screens all columns in the
factors DataFrame and selects those with positive objective lift.

In [ ]:
opt_auto = RatebookOptimiser(
    objective="expected_income",
    constraints={"volume": {"min": 0.90}},
    # factor_columns=None  → auto-discover
    max_cd_iterations=2,
    max_iter=50,
)

t0 = time.perf_counter()
result_auto = opt_auto.solve(df, factors)
elapsed = time.perf_counter() - t0

print(f"Auto-discover solve: {elapsed:.2f}s")
print(f"  Discovered factors: {list(result_auto.factor_tables.keys())}")
print(f"  Objective: {result_auto.total_objective:,.2f}")
print(f"  CD iterations: {result_auto.cd_iterations}")

## 7. MLflow-ready summary

In [ ]:
s = opt_multi.summary(result_multi)

print("=== params ===")
for k, v in s["params"].items():
    print(f"  {k}: {v!r}")

print("\n=== metrics ===")
for k, v in s["metrics"].items():
    print(f"  {k}: {v:.4f}")

print("\n=== artifacts ===")
print(f"  factor_tables: {list(s['artifacts']['factor_tables'].keys())}")
print(f"  rating_entries: {list(s['artifacts']['rating_entries'].keys())}")
for name, entry_df in s["artifacts"]["rating_entries"].items():
    print(f"    {name}: {entry_df.shape}")

## 8. Serialisation round-trip

Save the optimised lambdas via `ApplyOptimiser`, reload, and verify.

In [ ]:
import tempfile
from pathlib import Path

# Create an ApplyOptimiser with the solved lambdas
applier = pc.ApplyOptimiser(
    lambdas=result_multi.lambdas,
    objective="expected_income",
    constraints={"volume": {"min": 0.90}},
)

result_before = applier.apply(df)

with tempfile.TemporaryDirectory() as tmpdir:
    path = Path(tmpdir) / "ratebook_config.json"
    applier.save(path)
    print(f"Saved to: {path}")
    print(f"Contents: {path.read_text()[:200]}...")

    loaded = pc.ApplyOptimiser.load(path)

result_after = loaded.apply(df)

print(f"\nBefore save: {result_before.total_objective:,.2f}")
print(f"After load:  {result_after.total_objective:,.2f}")
print(f"Match: {abs(result_before.total_objective - result_after.total_objective) < 1e-6}")